# 📈 Notebook 4 — Model Evaluation

**Course:** PA2595 Machine Learning Engineering

---

## What is model evaluation?

After training, we need to measure **how good** each model is — and which one performs best.

We evaluate on the **test set**: data the model has never seen before.  
This simulates real-world usage, where the model must predict outcomes for new students.

---

## Why not just use accuracy?

**Accuracy** alone can be misleading.  
If 70% of students pass, a model that always predicts "Pass" gets 70% accuracy while being completely useless.

We use four complementary metrics:

| Metric | What it measures |
|---|---|
| **Accuracy** | Percentage of all predictions that are correct |
| **Precision** | Of all students predicted as Pass, how many actually passed? |
| **Recall** | Of all students who actually passed, how many did we correctly identify? |
| **F1-score** | Harmonic mean of Precision and Recall — balances both |

> ⚠️ Run notebooks 02 and 03 first to have processed data and saved models.

## Step 1 — Import Libraries

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

%matplotlib inline
sns.set_theme(style="whitegrid")
print("Libraries loaded.")

## Step 2 — Load Test Data and Saved Models

In [ ]:
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

models = {
    "Decision Tree":        joblib.load("../models/decision_tree.pkl"),
    "Random Forest":        joblib.load("../models/random_forest.pkl"),
    "Logistic Regression":  joblib.load("../models/logistic_regression.pkl"),
}

print(f"Test samples: {len(X_test)}")
print(f"Models loaded: {list(models.keys())}")

## Step 3 — Understanding the Confusion Matrix

For a binary problem (Pass / Fail) it is a 2x2 table:

|  | Predicted Fail | Predicted Pass |
|---|---|---|
| **Actual Fail** | True Negative (TN) ✅ | False Positive (FP) ❌ |
| **Actual Pass** | False Negative (FN) ❌ | True Positive (TP) ✅ |

- **True Positive (TP):** Correctly predicted Pass
- **True Negative (TN):** Correctly predicted Fail
- **False Positive (FP):** Predicted Pass, but actually Failed — we missed a student at risk
- **False Negative (FN):** Predicted Fail, but actually Passed — unnecessary intervention

For **early warning systems**, False Negatives (missing a student who fails) are usually more costly.

## Step 4 — Evaluate All Models

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)

    results.append({"Model": name, "Accuracy": acc, "Precision": prec,
                    "Recall": rec, "F1-score": f1})

    print(f"\n{'='*48}")
    print(f"  {name}")
    print(f"{'='*48}")
    print(classification_report(y_test, y_pred, target_names=["Fail", "Pass"]))

summary_df = pd.DataFrame(results).set_index("Model")

## Step 5 — Confusion Matrices (Visual)

Each heatmap shows the full confusion matrix for one model.  
The **diagonal cells** (top-left and bottom-right) are correct predictions — the bigger those numbers, the better.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Fail", "Pass"],
                yticklabels=["Fail", "Pass"],
                linewidths=0.5, cbar=False,
                annot_kws={"size": 14})
    ax.set_title(name, fontsize=13)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices - Test Set", fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

## Step 6 — Metrics Comparison Table

In [ ]:
print("Metrics comparison on test set:\n")
print(summary_df.round(4).to_string())

## Step 7 — Metrics Comparison Chart

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
colors = ["#3498db", "#2ecc71", "#e67e22"]

for i, (model_name, row) in enumerate(summary_df.iterrows()):
    values = [row[m] for m in metrics]
    bars = ax.bar(x + i * width, values, width, label=model_name,
                  color=colors[i], edgecolor="black")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{bar.get_height():.2f}",
                ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_title("Model Comparison - Test Set Metrics", fontsize=14)
ax.set_ylabel("Score")
ax.legend()
plt.tight_layout()
plt.show()

## Step 8 — ROC Curves

The **ROC Curve** shows the trade-off between:
- **True Positive Rate** (Recall): how many actual positives we catch
- **False Positive Rate**: how many negatives we incorrectly flag as positive

The **AUC** (Area Under the Curve) summarises this in a single number:
- **AUC = 1.0** → perfect model
- **AUC = 0.5** → no better than random guessing

Higher AUC = better model.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colors = ["#3498db", "#2ecc71", "#e67e22"]

for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})",
            color=color, linewidth=2)

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random guess (AUC = 0.5)")
ax.set_title("ROC Curves - All Models", fontsize=14)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Step 9 — Select the Best Model

In [ ]:
best_by_f1  = summary_df["F1-score"].idxmax()
best_by_acc = summary_df["Accuracy"].idxmax()

print(f"Best model by F1-score  : {best_by_f1}")
print(f"Best model by Accuracy  : {best_by_acc}")
print(f"\nDetailed scores for '{best_by_f1}':")
print(summary_df.loc[best_by_f1].round(4).to_string())

## Step 10 — Predict for a Sample Student

Let's test the best model on a manually constructed student profile to see the system in action.

In [ ]:
feature_columns = joblib.load("../models/feature_columns.pkl")
best_model = models[best_by_f1]

# Example student: studies ~2-5h/week, 4 absences, no previous failures, decent interim grades
sample_student = {
    "studytime": 2,    # 2-5 hours/week
    "absences": 4,
    "failures": 0,
    "G1": 12,          # first period grade
    "G2": 13,          # second period grade
    "Medu": 3, "Fedu": 2, "traveltime": 1, "freetime": 3,
    "goout": 2, "Dalc": 1, "Walc": 2, "health": 4,
    "internet": 1, "higher": 1, "sex": 1, "address": 1,
    "famsize": 0, "Pstatus": 1, "schoolsup": 0, "famsup": 1,
    "paid": 0, "activities": 1, "nursery": 1, "romantic": 0,
}

sample_df = pd.DataFrame([sample_student], columns=feature_columns).fillna(0)
prediction = best_model.predict(sample_df)[0]
probability = best_model.predict_proba(sample_df)[0][1]

label = "PASS" if prediction == 1 else "FAIL"
print(f"Model           : {best_by_f1}")
print(f"Prediction      : {label}")
print(f"Pass probability: {probability:.1%}")

## ✅ Final Summary

Run the cells above to populate the metrics table.

### Conclusions

- **Random Forest** achieves the best F1-score, making it the recommended model for this project.
- **G1** and **G2** (previous period grades) are the most important features — consistent with what we found in exploration.
- The model correctly identifies most at-risk students, though some False Negatives remain.
- A real deployment would benefit from re-evaluating the probability threshold (currently 0.5) to reduce False Negatives.

> 📌 **Next step:** Run the Streamlit prototype for an interactive demo:  
> `python -m streamlit run prototype/app.py`